# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jaineshchaurasiya20/FlyRank_Ml_Assignment/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Task type: ranking/scoring.** The decision is: *which content items should an editor refresh first?* I would produce a refresh-priority score for every page and rank the queue, rather than make a brittle yes/no decision. The score should combine the page's current opportunity, recent performance, content age, and search context. The output is for decision support: editors review the highest-priority pages first.

In [1]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("../../data/raw/content_refresh_anonymized.csv")
raw = pd.read_csv(DATA_PATH)

print(f"Loaded {len(raw):,} rows and {raw.shape[1]} columns.")
print(f"Unique content items: {raw['content_id'].nunique():,}")
print(f"Unique clients: {raw['client_id'].nunique():,}")

Loaded 30,000 rows and 44 columns.
Unique content items: 30,000
Unique clients: 32


## 2. Target or proxy

The final target should be an **observed future outcome**, such as `future_30d_impression_lift` or `future_30d_click_lift` after a refresh. A useful binary version would be `successful_refresh = 1` when a refreshed page beats its pre-refresh baseline in the agreed future window. That target is not present in this starter snapshot, so this notebook only sketches its shape and does not claim to train it.

`trend_direction` and `trend_pct` are available retrospective fields, but they are rule-derived from the same 30-day comparison and must not be treated as honest model features. They can be used only as a clearly labeled exploratory proxy until a future outcome is collected.

In [2]:
# Sketch the future target columns that a refresh experiment would add.
target_sketch = raw[["content_id", "client_id"]].copy()
target_sketch["future_30d_impression_lift"] = pd.NA
target_sketch["successful_refresh"] = pd.NA

target_sketch.head()

,content_id,client_id,future_30d_impression_lift,successful_refresh
0,content_304f48230142,client_f369cb89fc,<NA>,<NA>
1,content_a1fb4e703a9e,client_4e07408562,<NA>,<NA>
2,content_9aa793d4d895,client_7f2253d7e2,<NA>,<NA>
3,content_331d6c4de07b,client_19581e27de,<NA>,<NA>
4,content_d99b7a2d90ca,client_3fdba35f04,<NA>,<NA>


## 3. Success metric

**Primary metric: precision@K**, where K is the number of pages an editor can realistically refresh in one cycle. Good means that a high share of the top-K recommendations later show the agreed positive refresh outcome. This matches the queue decision and avoids pretending that every page can be acted on.

The metric must be measured on a future, held-out outcome window and compared with a simple baseline such as sorting by current impression volume or a transparent age/opportunity rule. Offline ranking quality is decision support, not proof that the refresh caused the improvement.

In [3]:
K = 100
print(f"A production evaluation would use precision@{K}.")
print("The starter data has no future refresh outcome, so precision cannot be computed honestly yet.")
print(f"Current retrospective down-trend share (exploratory only): {(raw['trend_direction'] == 'down').mean():.1%}")

A production evaluation would use precision@100.
The starter data has no future refresh outcome, so precision cannot be computed honestly yet.
Current retrospective down-trend share (exploratory only): 54.2%


## 4. The unit of analysis, as a real dataframe

**One row = one pseudonymized content item (page) at the export snapshot.** `client_id` identifies the client for grouping and later client-held-out validation; it is not a feature. The available columns describe the page, its search context, and trailing performance. A future label would be joined to this same content-item row after the refresh window closes.

In [4]:
display_columns = [
    "content_id", "client_id", "content_type", "main_intent",
    "impressions_90d", "clicks_90d", "sessions_90d",
    "impressions_last_30d", "impressions_prev_30d",
    "content_age_days", "days_since_last_update", "avg_position",
]
content_items = raw[display_columns].copy()

print("Unit check: one row = one content item")
print(f"Rows: {len(content_items):,} | Unique content_id values: {content_items['content_id'].nunique():,}")
display(content_items.head(8))

print("Missingness by content type for two fields that need care:")
missing_by_type = raw.groupby("content_type")[["word_count", "search_volume"]].apply(lambda frame: frame.isna().mean())
display(missing_by_type)

Unit check: one row = one content item
Rows: 30,000 | Unique content_id values: 30,000


,content_id,client_id,content_type,main_intent,impressions_90d,clicks_90d,sessions_90d,impressions_last_30d,impressions_prev_30d,content_age_days,days_since_last_update,avg_position
0,content_304f48230142,client_f369cb89fc,keyword article,transactional,3803,29,17,578,987,187,20,10.6
1,content_a1fb4e703a9e,client_4e07408562,keyword article,informational,15320,7,9,2501,5915,445,25,20.3
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,informational,12581,11,11,2382,6089,141,20,36.5
3,content_331d6c4de07b,client_19581e27de,keyword article,commercial,11751,58,78,3626,4206,463,22,6.2
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,informational,19140,24,145,4211,6452,263,14,44.0
5,content_d4084a4bc775,client_f369cb89fc,keyword article,transactional,3970,1,5,617,1009,147,20,8.5
6,content_9a34b442b552,client_8722616204,keyword article,informational,20,0,1,1,13,90,20,7.0
7,content_a63219c6e95a,client_19581e27de,keyword article,commercial,1724,1,28,636,632,445,22,21.2


Missingness by content type for two fields that need care:


,word_count,search_volume
content_type,,
comparison article,0.000000,0.000000
feedly article,0.000000,1.000000
keyword article,0.282979,0.013673


## 5. Why ML beats a fixed rule here

A fixed rule such as “refresh every page with fewer than 1,000 impressions” ignores interactions among age, intent, search demand, position, clicks, traffic quality, and missing-data patterns. Those signals can point in different directions, and their useful weights may change by client and content type. A model can learn a ranking from many historical examples and be checked against future outcomes.

ML is justified only if it beats the rule baseline on a held-out client/time split and produces a queue editors can act on. Wrong calls have asymmetric costs: a false positive wastes an editor's time, while a false negative can leave a valuable page declining. The output therefore supports review and prioritization, not automatic publishing.

In [5]:
# A simple fixed-rule comparator can be defined now, even though the future label is not.
# This makes the later ML-vs-rule comparison explicit without pretending it is evaluated here.
rule_queue = content_items.sort_values(
    ["days_since_last_update", "impressions_90d"],
    ascending=[False, True],
).head(K)

print(f"Example rule queue contains {len(rule_queue)} content items.")
print("Later, compare its precision@K with the model-ranked queue on future observed outcomes.")
display(rule_queue[["content_id", "content_type", "impressions_90d", "days_since_last_update"]].head())

Example rule queue contains 100 content items.
Later, compare its precision@K with the model-ranked queue on future observed outcomes.


,content_id,content_type,impressions_90d,days_since_last_update
4606,content_3f3576c295f5,keyword article,1,373
29384,content_f6fdf87348f6,keyword article,2,373
26242,content_55a5b1c46474,keyword article,35,373
18440,content_8d56efff1e71,keyword article,1,372
24216,content_1b4ec72dafd4,keyword article,2,372


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.